### Study 1: Merge individual-level DFA results, cross correlations, pseudo cross correlations with OIRs

In [1]:
import pandas as pd

# Load datasets: individual-level, cross correlations, pseudo cross correlations, oirs
ind = pd.read_csv("results/study1/study1_dfa_rms.csv")
cross = pd.read_csv("results/study1/study1_dfa_rms_corr_with_noise.csv")
pseudo_cross = pd.read_csv("results/study1/study1_dfa_rms_pseudo_corr_with_noise.csv")
oir_ind = pd.read_csv("final datasets/study1/oirs_study1_individual.csv")
oir_pair = pd.read_csv("final datasets/study1/oirs_study1_group.csv")
# Pivot individual-level data longer
ind_long = (
    ind.melt(
        id_vars=[c for c in ind.columns if not c.endswith(("_p1", "_p2"))],
        value_vars=["head_mag_rms_p1", "head_mag_rms_p2",
                    "head_mag_alpha_p1", "head_mag_alpha_p2"],
        var_name="measure_subject",
        value_name="value"
    )
    .assign(
        subject=lambda d: d["measure_subject"].str.extract(r"(p\d)"),
        measure=lambda d: d["measure_subject"].str.replace(r"_p\d", "", regex=True)
    )
    .pivot_table(
        index=[c for c in ind.columns if not c.endswith(("_p1", "_p2"))] + ["subject"],
        columns="measure",
        values="value"
    )
    .reset_index()
)

# Standardise pseudo columns (remove 'pseudo_' prefix) for merging
pseudo_cross = pseudo_cross.rename(
    columns={col: col.replace("pseudo_", "") for col in pseudo_cross.columns}
)

# Add a column to distinguish actual vs pseudo
cross["pair_type"] = "actual"
pseudo_cross["pair_type"] = "pseudo"

# Row-bind actual and pseudo
cross_all = pd.concat([cross, pseudo_cross], ignore_index=True)

# Map speaker and subject for merging
oir_ind = oir_ind.assign(
    subject=oir_ind["speaker"].map({"left": "p1", "right": "p2"})
)

# Merge with OIRs
ind_oir = oir_ind.merge(ind_long, on=["pair", "condition", "environment", "subject"], how="left")
cross_oir = oir_pair.merge(cross_all, on=["pair", "condition", "environment"], how="left")

# Save merged files
ind_oir.to_csv("results/study1/study1_dfa_models.csv", index=False)
cross_oir.to_csv("results/study1/study1_cc_models.csv", index=False)

print("Saved all merged files successfully!")

Saved all merged files successfully!


### Study 2: Merge individual-level DFA results, cross correlations, pseudo cross correlations with OIRs

In [2]:
import pandas as pd

# Load datasets: individual-level, cross correlations, pseudo cross correlations, oirs
ind = pd.read_csv("results/study2/study2_dfa_rms.csv")
cross = pd.read_csv("results/study2/study2_dfa_rms_corr_with_noise.csv")
pseudo_cross = pd.read_csv("results/study2/study2_dfa_rms_pseudo_corr_with_noise.csv")
oir = pd.read_csv("final datasets/study2/oirs_study2.csv")

# Recode role so that left = 1 and right = 2
ind["role"] = ind["role"].map({"Left": "1", "Right": "2"})

# Standardise pseudo columns (remove 'pseudo_' prefix) for merging
pseudo_cross = pseudo_cross.rename(
    columns={col: col.replace("pseudo_", "") for col in pseudo_cross.columns}
)

# Add a column to distinguish actual vs pseudo
cross["pair_type"] = "actual"
pseudo_cross["pair_type"] = "pseudo"

# Row-bind actual and pseudo
cross_all = pd.concat([cross, pseudo_cross], ignore_index=True)

# Standardise column data types for merging
ind["role"] = ind["role"].astype(int)
oir["role"] = oir["role"].astype(int)

# Collapse OIRs to pair-level by averaging across roles
oir_pair = (
    oir.groupby(["pair", "trial"], as_index=False)
       .sum(numeric_only=True)   # or .agg(...) if you want a different rule
)

# Merge with OIRs
ind_oir = ind.merge(oir, on=["pair", "trial", "role"], how="left")
cross_oir = cross_all.merge(oir_pair, on=["pair", "trial"], how="left")

# Save merged files
ind_oir.to_csv("results/study2/study2_dfa_models.csv", index=False)
cross_oir.to_csv("results/study2/study2_cc_models.csv", index=False)

print("Saved all merged files successfully!")

Saved all merged files successfully!
